# E蛋白主动学习工作流：突变体生成与不确定性评估 (v3 修正版)

--- 

**目标**: 本工作流旨在通过计算方法高效探索E蛋白的突变空间，为湿实验提供数据驱动的指导。

**版本说明**: 此版本已根据用户反馈进行了重要修正，确保数据预处理流程与参考代码(`Site-Saturation_Mutagenesis.ipynb`)完全一致。

1.  **生成候选突变体池**: 脚本将使用`ESMProtein`和`.encode()`方法，从FASTA和PDB文件动态生成准确的序列和结构token。然后，针对不同的背景序列（WT、T9I、T11A）进行饱和突变扫描，并将数据保存为pickle文件。
2.  **模型推理与不确定性评估**: 加载指定的`mc_dropout`模型检查点，对生成的所有突变体进行批量推理。计算每个突变体对三个性质（Expression, CCK8, Activation）的预测值和不确定性，并汇总到指定的Excel表格中。

---

## 步骤一：生成候选突变体数据池 (修正后)

此单元格的核心逻辑已重写，以确保正确使用ESM API进行tokenization。

In [3]:
# 导入必要的库
import pickle
import os
import json
import torch
from tqdm.notebook import tqdm

# 导入您的项目模块
import ioutils
import trainUtils
from esm.sdk.api import ESMProtein

# --- 💡 用户配置区 --- #

# 1. 输入文件路径
FASTA_PATH = "fasta/E.fasta"
PDB_PATH = "pdb/E.pdb"

# 2. 用于编码PDB文件的模型配置文件路径 (仅用于获取结构token)
ENCODER_CONFIG_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/k3/config.json"

# 3. 输出目录
OUTPUT_DIR = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 辅助函数 --- #
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

def generate_saturation_mutagenesis(base_sequence: str, background_name: str):
    """对给定的基础序列进行饱和突变扫描"""
    mutants = []
    seq_list = list(base_sequence)
    for i in range(len(base_sequence)):
        original_aa = seq_list[i]
        for new_aa in AMINO_ACIDS:
            if original_aa == new_aa:
                continue
            
            mutant_seq_list = seq_list[:]
            mutant_seq_list[i] = new_aa
            mutant_sequence = "".join(mutant_seq_list)
            mutation_id = f"{background_name}_{original_aa}{i+1}{new_aa}"
            
            mutants.append({
                "id": mutation_id,
                "sequence": mutant_sequence
            })
    return mutants

# --- 主逻辑 --- #
print("开始生成突变体数据...")

try:
    # 1. 加载用于编码的基础ESM模型 (与您的参考代码一致)
    with open(ENCODER_CONFIG_PATH, 'r') as f:
        encoder_configs = json.load(f)
    esm_encoder_model = trainUtils.loadPretrainModel(encoder_configs)
    esm_encoder_model.eval()
    print("ESM编码器模型加载成功。")

    # 2. 从FASTA读取WT序列
    fasta_gen = ioutils.readFasta(FASTA_PATH)
    _, WT_SEQUENCE = next(fasta_gen)
    print(f"从 '{FASTA_PATH}' 加载WT序列成功，长度: {len(WT_SEQUENCE)}")

    # 3. 创建参考蛋白对象并获取参考tokens (序列和结构)
    ref_protein_obj = ESMProtein.from_pdb(PDB_PATH)
    with torch.no_grad():
        encoded_ref = esm_encoder_model.encode(ref_protein_obj)
        ref_seq_tokens = encoded_ref.sequence
        ref_structure_tokens = encoded_ref.structure
    print(f"从 '{PDB_PATH}' 加载并编码参考序列和结构成功。")

    # 4. 定义突变背景
    MUTATION_BACKGROUNDS = {
        "E_WT": WT_SEQUENCE,
        "E_T9I": WT_SEQUENCE[:8] + "I" + WT_SEQUENCE[9:],
        "E_T11A": WT_SEQUENCE[:10] + "A" + WT_SEQUENCE[11:],
    }

    # 5. 循环处理每个突变背景
    for name, sequence in MUTATION_BACKGROUNDS.items():
        print(f"\n--- 正在处理背景: {name} ---")
        
        mutants_info = generate_saturation_mutagenesis(sequence, name)
        print(f"为 {name} 背景生成了 {len(mutants_info)} 个单点突变体。")
        
        unlabeled_pool = []
        for mutant in tqdm(mutants_info, desc=f"Tokenizing {name}"):
            # 为每个突变体序列创建对象并编码，以获取正确的序列token
            mutant_protein_obj = ESMProtein(sequence=mutant['sequence'])
            with torch.no_grad():
                encoded_mutant = esm_encoder_model.encode(mutant_protein_obj)
                mutant_seq_tokens = encoded_mutant.sequence
            
            sample = {
                "id": mutant["id"],
                "E": {
                    "seq_t": mutant_seq_tokens,       # 突变体序列token
                    "structure_t": ref_structure_tokens, # 参考结构token
                },
                "aligned_E": { 
                    "seq_t": ref_seq_tokens,           # 参考序列token
                    "structure_t": ref_structure_tokens, # 参考结构token
                }
            }
            unlabeled_pool.append(sample)
            
        output_path = os.path.join(OUTPUT_DIR, f"{name}_saturation_mutants.pkl")
        with open(output_path, "wb") as f:
            pickle.dump(unlabeled_pool, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"已将 {len(unlabeled_pool)} 个突变体数据保存到: {output_path}")

    print("\n所有突变池生成完毕！")

except Exception as e:
    print(f"\n发生错误: {e}")
    print("请检查所有文件路径是否正确，并且相关文件存在。")


开始生成突变体数据...
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
ESM编码器模型加载成功。
从 'fasta/E.fasta' 加载WT序列成功，长度: 75
从 'pdb/E.pdb' 加载并编码参考序列和结构成功。

--- 正在处理背景: E_WT ---
为 E_WT 背景生成了 1425 个单点突变体。


Tokenizing E_WT:   0%|          | 0/1425 [00:00<?, ?it/s]

已将 1425 个突变体数据保存到: /data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_WT_saturation_mutants.pkl

--- 正在处理背景: E_T9I ---
为 E_T9I 背景生成了 1425 个单点突变体。


Tokenizing E_T9I:   0%|          | 0/1425 [00:00<?, ?it/s]

已将 1425 个突变体数据保存到: /data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_T9I_saturation_mutants.pkl

--- 正在处理背景: E_T11A ---
为 E_T11A 背景生成了 1425 个单点突变体。


Tokenizing E_T11A:   0%|          | 0/1425 [00:00<?, ?it/s]

已将 1425 个突变体数据保存到: /data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_T11A_saturation_mutants.pkl

所有突变池生成完毕！


In [4]:
with open("/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_T11A_saturation_mutants.pkl", 'rb') as f:
    loaded_data = pickle.load(f)
loaded_data

[{'id': 'E_T11A_M1A',
  'E': {'seq_t': tensor([ 0,  5, 19,  8, 18,  7,  8,  9,  9, 11,  6,  5,  4, 12,  7, 17,  8,  7,
            4,  4, 18,  4,  5, 18,  7,  7, 18,  4,  4,  7, 11,  4,  5, 12,  4, 11,
            5,  4, 10,  4, 23,  5, 19, 23, 23, 17, 12,  7, 17,  7,  8,  4,  7, 15,
           14,  8, 18, 19,  7, 19,  8, 10,  7, 15, 17,  4, 17,  8,  8, 10,  7, 14,
           13,  4,  4,  7,  2]),
   'structure_t': tensor([4098, 2048,  264, 2439,  137,  264, 1197, 2048, 3056,  264,  264,  137,
            264, 3961, 2048, 1197, 2056, 1476, 1197,  588,  123,  588,  588, 1450,
            264, 1476, 1450,  588, 1476, 2048, 1197,  588,  588, 2048, 1197,  588,
            588, 1476, 1197,  588,  588, 1476,  588,  588, 1197,  588, 1197, 1476,
           1197,  445, 2983, 3407, 2048,  137, 2156, 1197,  588, 1197, 2048,  588,
            588,  987, 1476,  588, 1197, 2048, 1197,  123, 1197, 2439, 1892, 2093,
           1265,  264,  264,  264, 4097])},
  'aligned_E': {'seq_t': tensor([ 0, 20, 1

## 步骤二：模型推理与不确定性评估

此单元格将加载上一步生成的所有`.pkl`文件，调用您训练好的模型进行推理，并计算每个突变体的预测值和不确定性。

In [1]:
# 导入必要的库
import torch
import numpy as np
import pandas as pd
import json
import pickle
import glob
import os
import sys
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader

# 导入您的项目模块
import trainUtils
from VirusDataset import VESMDataset

# --- 💡 用户配置区 --- #

# 1. 模型和配置路径
CONFIG_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/k3/config.json"
CHECKPOINT_PATH = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/k3/epoch=717-validation_pearson=2.1272.ckpt"

# 2. 突变体数据池路径 (使用通配符 * 来匹配所有由上一步生成的pkl文件)
MUTANT_POOL_GLOB_PATTERN = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_*_saturation_mutants.pkl"

# 3. 推理参数
DEVICE = "cuda:4" if torch.cuda.is_available() else "cpu"
N_FORWARD_PASSES = 50  # MC Dropout的前向传播次数
BATCH_SIZE = 1        # 批处理大小，根据您的GPU显存调整

# 4. 输出的Excel文件名
OUTPUT_EXCEL_PATH = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/mutant_uncertainty_results.xlsx"
os.makedirs(os.path.dirname(OUTPUT_EXCEL_PATH), exist_ok=True)

# 5. 预测值的名称 (需要和模型回归头的输出顺序完全一致!)
PREDICTION_NAMES = ["Expression", "CCK8", "Activation"]

# --- 主逻辑 --- #
model = None # 初始化变量以进行错误检查

# 1. 加载模型
print("--- 步骤1: 加载模型 ---")
try:
    with open(CONFIG_PATH, "r") as f:
        configs = json.load(f)
    
    configs['train']['batch_size'] = BATCH_SIZE

    pretrain_model = trainUtils.loadPretrainModel(configs)
    model = trainUtils.buildModel(configs, pretrain_model, CHECKPOINT_PATH)
    model.to(DEVICE)
    model.eval()

    if configs["model"]["params"].get("regressor_version") == "mc_dropout":
        print("INFO: MC Dropout已开启，用于不确定性评估。")
        model.enable_mc_dropout()
    else:
        print("警告: 模型配置非'mc_dropout'模式。不确定性评估可能无意义。")
    print(f"模型已加载到 {DEVICE}")

except FileNotFoundError:
    print(f"错误: 找不到模型配置文件 '{CONFIG_PATH}' 或模型权重 '{CHECKPOINT_PATH}'。请检查路径。")
except Exception as e:
    print(f"加载模型时发生未知错误: {e}")



--- 步骤1: 加载模型 ---
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
model at stage: training stage 1
INFO: Using Regressors with MC Dropout for new model.


/home/zhoukaitao/anaconda3/envs/esm3/lib/python3.10/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


incompatible parameters ['stage_1_regressors.clf.3.weight', 'stage_1_regressors.clf.3.bias', 'stage_2_regressors.clf.3.weight', 'stage_2_regressors.clf.3.bias']
finish loading parameters
load model from checkpoint /data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/k3/epoch=717-validation_pearson=2.1272.ckpt
INFO: MC Dropout已开启，用于不确定性评估。
模型已加载到 cuda:4


In [2]:
# 2. 加载和合并所有突变体数据
BATCH_SIZE = 256

if model is not None:
    print("\n--- 步骤2: 加载所有突变体数据 ---")
    all_mutant_files = glob.glob(MUTANT_POOL_GLOB_PATTERN)
    if not all_mutant_files:
        print(f"错误: 在'{MUTANT_POOL_GLOB_PATTERN}'下找不到任何突变体数据文件。")
    else:
        print(f"找到 {len(all_mutant_files)} 个突变体文件:")
        all_unlabeled_data = []
        for file_path in all_mutant_files:
            print(f"  - {os.path.basename(file_path)}")
            with open(file_path, "rb") as f:
                all_unlabeled_data.extend(pickle.load(f))
        
        print(f"总共加载了 {len(all_unlabeled_data)} 个突变体进行评估。")

        # 3. 创建DataLoader
        unlabeled_dataset = VESMDataset(
            [all_unlabeled_data],
            stage="inference",
            seq=configs["dataset"]["seq"],
            train_time_series=False,
        )
        unlabeled_loader = DataLoader(
            unlabeled_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=os.cpu_count() // 2 if os.cpu_count() > 1 else 0,
        )

        # 4. 进行MC Dropout推理
        print("\n--- 步骤3: 开始MC Dropout推理 ---")
        all_predictions = []
        all_ids = []

        with torch.no_grad():
            for batch in tqdm(unlabeled_loader, desc="推理进度"):
                input_data = batch["input"]
                for k, v in input_data.items():
                    if isinstance(v, dict):
                        for vk, vv in v.items():
                            # 将tensor列表转换为torch tensor
                            if isinstance(vv, list):
                                v[vk] = torch.tensor(vv, dtype=torch.long).to(DEVICE)
                            elif isinstance(vv, torch.Tensor):
                                v[vk] = vv.to(DEVICE)
                    elif isinstance(v, torch.Tensor):
                        input_data[k] = v.to(DEVICE)

                batch_preds_list = []
                for _ in range(N_FORWARD_PASSES):
                    output = model.forward(input_data)
                    preds = output.S1Predicts['E']['predictions']
                    batch_preds_list.append(preds.cpu())

                batch_preds_tensor = torch.stack(batch_preds_list)
                all_predictions.append(batch_preds_tensor)
                all_ids.extend(batch["meta"]["id"])

        # 5. 计算均值和不确定性并整理结果
        print("\n--- 步骤4: 计算结果并保存到Excel ---")
        
        predictions_tensor = torch.cat(all_predictions, dim=1)
        mean_preds = predictions_tensor.mean(dim=0).numpy()
        uncertainty_scores = predictions_tensor.var(dim=0).numpy()

        results_df = pd.DataFrame({
            'mutation_id': all_ids,
            'background': [mid.split('_')[1] for mid in all_ids]
        })
        
        for i, name in enumerate(PREDICTION_NAMES):
            results_df[f'{name}_pred'] = mean_preds[:, i]
            results_df[f'{name}_uncertainty'] = uncertainty_scores[:, i]
        
        results_df['total_uncertainty'] = uncertainty_scores.sum(axis=1)
        results_df = results_df.sort_values(by='total_uncertainty', ascending=False).reset_index(drop=True)
        
        results_df.to_excel(OUTPUT_EXCEL_PATH, index=False, engine='openpyxl')
        print(f"成功！结果已保存到 '{OUTPUT_EXCEL_PATH}'")
        
        print("\n结果预览:")
        display(results_df.head())


--- 步骤2: 加载所有突变体数据 ---
找到 3 个突变体文件:
  - E_WT_saturation_mutants.pkl
  - E_T11A_saturation_mutants.pkl
  - E_T9I_saturation_mutants.pkl
总共加载了 4275 个突变体进行评估。

--- 步骤3: 开始MC Dropout推理 ---


推理进度:   0%|          | 0/34 [00:00<?, ?it/s]


--- 步骤4: 计算结果并保存到Excel ---
成功！结果已保存到 '/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/mutant_uncertainty_results.xlsx'

结果预览:


,mutation_id,background,Expression_pred,Expression_uncertainty,CCK8_pred,CCK8_uncertainty,Activation_pred,Activation_uncertainty,total_uncertainty
0,E_T11A_S55I,T11A,4.986306,1.519331,2.046806,0.116272,1.537708,0.119184,1.754787
1,E_T9I_R69L,T9I,-6.127627,1.326833,1.602178,0.124638,1.369037,0.170501,1.621973
2,E_T11A_R69I,T11A,-7.962182,1.259096,1.371841,0.145518,1.285339,0.179951,1.584565
3,E_WT_R69V,WT,-7.943394,1.239082,1.151590,0.143398,1.138438,0.162413,1.544893
4,E_WT_R69F,WT,-6.544693,1.208267,1.521064,0.135439,1.283862,0.140507,1.484213


In [1]:
import torch
import numpy as np
import pandas as pd
import json
import pickle
import glob
import os
import re
import shutil
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
import torch.multiprocessing as mp

# 导入您的项目模块
import trainUtils
from VirusDataset import VESMDataset

# --- 💡 用户配置区 --- #
BASE_CHECKPOINT_DIR = "/data2/zhoukaitao/01evoModel/checkpoints/251015_E_Drop/"
FOLDS = ["k1", "k2", "k3", "k4"] # 交叉验证文件夹
MUTANT_POOL_GLOB_PATTERN = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/E_*_saturation_mutants.pkl"
OUTPUT_EXCEL_PATH = "/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/ensemble_mutant_uncertainty_results.xlsx"

# --- 推理参数 --- #
N_PASSES_PER_MODEL = 25
BATCH_SIZE = 300
DEVICES = [f"cuda:{i}" for i in [0,2,3,4]] # 为每个模型分配一个GPU
PREDICTION_NAMES = ["Expression", "CCK8", "Activation"]

# --- 辅助函数 --- #
def find_best_ckpt(fold_dir):
    """在指定文件夹中查找validation_pearson分数最高的ckpt文件""" 
    ckpt_files = glob.glob(os.path.join(fold_dir, "*.ckpt"))
    if not ckpt_files:
        raise FileNotFoundError(f"在 '{fold_dir}' 中找不到任何 .ckpt 文件。")
    
    best_ckpt = None
    max_pearson = -float('inf')
    
    for ckpt in ckpt_files:
        match = re.search(r"validation_pearson=([\d\.]+)\.ckpt", os.path.basename(ckpt))
        if match:
            pearson_score = float(match.group(1))
            if pearson_score > max_pearson:
                max_pearson = pearson_score
                best_ckpt = ckpt
                
    if best_ckpt is None:
        raise ValueError(f"在 '{fold_dir}' 的文件名中找不到 'validation_pearson' 分数。")
        
    return best_ckpt, max_pearson

def inference_worker(rank, fold, config_path, best_ckpt_path, all_unlabeled_data, temp_dir):
    """并行推理的工作进程函数，将结果保存到文件"""
    device = DEVICES[rank]
    from tqdm import tqdm
    print(f"进程 {rank} ({fold}): 开始在 {device} 上进行推理...")
    
    try:
        with open(config_path, "r") as f:
            configs = json.load(f)
        
        pretrain_model = trainUtils.loadPretrainModel(configs)
        model = trainUtils.buildModel(configs, pretrain_model, best_ckpt_path)
        model.to(device)
        model.eval()
        if configs["model"]["params"].get("regressor_version") == "mc_dropout":
            model.enable_mc_dropout()

        dataset = VESMDataset([all_unlabeled_data], "inference", configs["dataset"]["seq"], train_time_series=False)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        fold_predictions = []
        with torch.no_grad():
            for _ in tqdm(range(N_PASSES_PER_MODEL), desc=f'{fold} on {device}', position=rank, leave=False):
                pass_predictions = []
                for batch in loader:
                    input_data = batch["input"]
                    for k, v in input_data.items():
                        if isinstance(v, dict):
                            for vk, vv in v.items():
                                v[vk] = torch.tensor(vv, dtype=torch.long).to(device) if isinstance(vv, list) else vv.to(device)
                    
                    output = model.forward(input_data)
                    preds = output.S1Predicts['E']['predictions']
                    pass_predictions.append(preds.cpu())
                fold_predictions.append(torch.cat(pass_predictions, dim=0))
        
        all_ids = [sample['id'] for sample in all_unlabeled_data]
        output_path = os.path.join(temp_dir, f"{fold}_results.pt")
        torch.save({'preds': torch.stack(fold_predictions), 'ids': all_ids}, output_path)
        print(f"进程 {rank} ({fold}): 推理完成，结果已保存。")
    except Exception as e:
        import traceback
        print(f"进程 {rank} ({fold}) 发生错误: {e}\n{traceback.format_exc()}")

# --- 主编排逻辑 --- #
if __name__ == '__main__':
    # try:
    #     mp.set_start_method('spawn', force=True)
    #     print("Multiprocessing start method set to 'spawn'.")
    # except RuntimeError:
    #     pass
        
    print("--- 步骤A: 加载所有突变体数据 ---")
    all_mutant_files = glob.glob(MUTANT_POOL_GLOB_PATTERN)
    if not all_mutant_files:
        print(f"错误: 在'{MUTANT_POOL_GLOB_PATTERN}'下找不到任何突变体数据文件。请先运行步骤一。")
    else:
        all_unlabeled_data = []
        for file_path in sorted(all_mutant_files):
            print(f"  - 加载 {os.path.basename(file_path)}")
            with open(file_path, "rb") as f:
                all_unlabeled_data.extend(pickle.load(f))
        print(f"总共加载了 {len(all_unlabeled_data)} 个突变体进行评估。")
        
        print("\n--- 步骤B: 准备并行推理任务 ---")
        processes = []
        temp_output_dir = "./temp_inference_results"
        if os.path.exists(temp_output_dir): shutil.rmtree(temp_output_dir)
        os.makedirs(temp_output_dir)
        
        for rank, fold in enumerate(FOLDS):
            fold_dir = os.path.join(BASE_CHECKPOINT_DIR, fold)
            config_path = os.path.join(fold_dir, 'config.json')
            best_ckpt, score = find_best_ckpt(fold_dir)
            print(f"  - {fold}: 找到最佳ckpt '{os.path.basename(best_ckpt)}' (pearson={score:.4f})")
            
            p = mp.Process(target=inference_worker, args=(rank, fold, config_path, best_ckpt, all_unlabeled_data, temp_output_dir))
            processes.append(p)
            
        print("\n--- 步骤C: 启动并行推理 (这可能需要较长时间) ---")
        for p in processes:
            p.start()
        
        # 等待所有进程完成
        for p in tqdm(processes, desc="等待模型完成"):
            p.join()
        
        print("\n--- 步骤D: 合并结果并计算统计数据 ---")
        result_files = glob.glob(os.path.join(temp_output_dir, "*_results.pt"))
        if len(result_files) != len(FOLDS):
            print(f"\n错误: 预期有 {len(FOLDS)} 个结果文件，但只找到 {len(result_files)} 个。请检查上面的错误信息。")
        else:
            all_preds_list = []
            final_ids = None
            for res_file in sorted(result_files):
                data = torch.load(res_file)
                all_preds_list.append(data['preds'])
                if final_ids is None: final_ids = data['ids']
            
            final_preds_tensor = torch.cat(all_preds_list, dim=0)
            mean_preds = final_preds_tensor.mean(dim=0).numpy()
            variance_preds = final_preds_tensor.var(dim=0).numpy()

            results_df = pd.DataFrame({
                'mutation_id': final_ids,
                'background': [mid.split('_')[1] for mid in final_ids]
            })
            
            for i, name in enumerate(PREDICTION_NAMES):
                results_df[f'{name}_pred_mean'] = mean_preds[:, i]
                results_df[f'{name}_uncertainty_var'] = variance_preds[:, i]
            
            results_df['total_uncertainty_var'] = variance_preds.sum(axis=1)
            results_df = results_df.sort_values(by='total_uncertainty_var', ascending=False).reset_index(drop=True)
            
            results_df.to_excel(OUTPUT_EXCEL_PATH, index=False, engine='openpyxl')
            print(f"成功！最终结果已保存到 '{OUTPUT_EXCEL_PATH}'")
            
            print("\n结果预览:")
            display(results_df.head())
            
            # 清理临时文件 (已注释掉)
            # shutil.rmtree(temp_output_dir)
            print(f"临时推理结果已保留在文件夹: '{temp_output_dir}'")

--- 步骤A: 加载所有突变体数据 ---
  - 加载 E_T11A_saturation_mutants.pkl
  - 加载 E_T9I_saturation_mutants.pkl
  - 加载 E_WT_saturation_mutants.pkl
总共加载了 4275 个突变体进行评估。

--- 步骤B: 准备并行推理任务 ---
  - k1: 找到最佳ckpt 'epoch=235-validation_pearson=1.8864.ckpt' (pearson=1.8864)
  - k2: 找到最佳ckpt 'epoch=1245-validation_pearson=2.0691.ckpt' (pearson=2.0691)
  - k3: 找到最佳ckpt 'epoch=717-validation_pearson=2.1272.ckpt' (pearson=2.1272)
  - k4: 找到最佳ckpt 'epoch=988-validation_pearson=2.1657.ckpt' (pearson=2.1657)

--- 步骤C: 启动并行推理 (这可能需要较长时间) ---
进程 0 (k1): 开始在 cuda:0 上进行推理...
进程 1 (k2): 开始在 cuda:2 上进行推理...
进程 2 (k3): 开始在 cuda:3 上进行推理...



等待模型完成:   0%|          | 0/4 [00:00<?, ?it/s]

进程 3 (k4): 开始在 cuda:4 上进行推理...load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
model at stage: training stage 1
model at stage: training stage 1
INFO: Using Regressors with MC Dropout for new model.


/home/zhoukaitao/anaconda3/envs/esm3/lib/python3.10/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


model at stage: training stage 1
model at stage: training stage 1
INFO: Using Regressors with MC Dropout for new model.


...

进程 3 (k4): 推理完成，结果已保存。


进程 0 (k1): 推理完成，结果已保存。


进程 1 (k2): 推理完成，结果已保存。

--- 步骤D: 合并结果并计算统计数据 ---
成功！最终结果已保存到 '/data2/zhoukaitao/01evoModel/dataset/251015_E_Saturat/ensemble_mutant_uncertainty_results.xlsx'

结果预览:


,mutation_id,background,Expression_pred_mean,Expression_uncertainty_var,CCK8_pred_mean,CCK8_uncertainty_var,Activation_pred_mean,Activation_uncertainty_var,total_uncertainty_var
0,E_T9I_R69V,T9I,-5.190882,17.574600,1.307528,0.140615,1.190469,0.208631,17.923847
1,E_T9I_R69I,T9I,-5.238085,15.884687,1.284759,0.218621,1.255216,0.199385,16.302692
2,E_T9I_R69F,T9I,-3.928689,11.198917,1.491889,0.123408,1.317897,0.135186,11.457512
3,E_T9I_R69L,T9I,-3.519322,10.833582,1.476944,0.079770,1.301024,0.105327,11.018680
4,E_T11A_R69V,T11A,-5.369326,10.582488,1.446563,0.168081,1.436126,0.157164,10.907734


临时推理结果已保留在文件夹: './temp_inference_results'
